# 🛡️ Multilingual Abuse Detection — Phase 2
## Hinglish + English | Multi-Label | Full Evaluation Pipeline

---
| Section | What it does |
|---|---|
| 1 | Install & imports |
| 2 | Config — **all hyperparams live here** |
| 3 | Drive mount + W&B |
| 4 | Checkpoint Manager (auto-resume across sessions) |
| 5 | Data download — Jigsaw (Kaggle) + HASOC (Hindi/Hinglish) |
| 6 | EDA + language distribution plots |
| 7 | Preprocessing — Hindi/Hinglish-aware pipeline |
| 8 | Train/Val split |
| 9 | Dataset class + DataLoaders |
| 10 | Model load (from Drive if exists, else HuggingFace) |
| 11 | Training — resume + early stopping (patience=5) + grad accum + AMP |
| 12 | Load best model |
| 13 | Threshold tuning per label |
| 14 | Full evaluation suite |
| 15 | Attention visualization |
| 16 | Interactive demo |


## Section 1 — Install

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 1 — INSTALL
# ═══════════════════════════════════════════════════════════
!pip install transformers torch datasets emoji langdetect lime wandb -q
!pip install scikit-learn matplotlib seaborn -q
!pip install indic-nlp-library -q   # Hindi normalisation
print("All packages installed ✅")

## Section 2 — Config  *(change only this cell)*

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 2 — CONFIG
# Every hyperparameter lives here. Swap MODEL_NAME to
# "xlm-roberta-base" at any time — nothing else changes.
# ═══════════════════════════════════════════════════════════

import os, json, re, zipfile, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
warnings.filterwarnings("ignore")

# ── Model ────────────────────────────────────────────────────
MODEL_NAME           = "google/muril-base-cased"   # or "xlm-roberta-base"
MAX_LEN              = 128
NUM_LABELS           = 6      # toxic | severe_toxic | obscene | threat | insult | identity_hate

# ── Training ─────────────────────────────────────────────────
BATCH_SIZE           = 16
GRAD_ACCUM_STEPS     = 2      # effective batch = 16×2 = 32
EPOCHS               = 10
LR                   = 2e-5
WEIGHT_DECAY         = 0.01
WARMUP_STEPS         = 500
EARLY_STOP_PATIENCE  = 5      # stop if val F1 doesn't improve for 5 epochs

# ── Paths ─────────────────────────────────────────────────────
DRIVE_ROOT           = "/content/drive/MyDrive/abuse_detection"
CHECKPOINT_DIR       = f"{DRIVE_ROOT}/checkpoints"
DATA_DIR             = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

# ── Misc ──────────────────────────────────────────────────────
SEED                 = 42
DEVICE               = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABEL_COLS           = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
LABEL_DISPLAY        = ["Toxic", "Severe Toxic", "Obscene", "Threat", "Insult", "Identity Hate"]

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Device : {DEVICE}")
print(f"Model  : {MODEL_NAME}")
print(f"Labels : {LABEL_COLS}")

## Section 3 — Drive Mount + W&B

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 3 — DRIVE MOUNT + W&B INIT
# ═══════════════════════════════════════════════════════════
from google.colab import drive, userdata
drive.mount("/content/drive")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DRIVE_ROOT, exist_ok=True)

import wandb
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
wandb.init(
    project = "multilingual-abuse-detection",
    name    = f"muril-multilingual-{datetime.now().strftime('%m%d-%H%M')}",
    config  = {
        "model":            MODEL_NAME,
        "batch_size":       BATCH_SIZE,
        "effective_batch":  BATCH_SIZE * GRAD_ACCUM_STEPS,
        "epochs":           EPOCHS,
        "lr":               LR,
        "max_len":          MAX_LEN,
        "early_stopping":   EARLY_STOP_PATIENCE,
        "num_labels":       NUM_LABELS,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
    }
)
print("W&B initialised ✅")

## Section 4 — Checkpoint Manager

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 4 — CHECKPOINT MANAGER
# Saves the full training state to Drive so you can
# close Colab and resume exactly where you left off.
# ═══════════════════════════════════════════════════════════

class CheckpointManager:
    """
    Saves/loads model weights + optimizer/scheduler/scaler
    states + epoch metadata to Google Drive.
    """

    def __init__(self, save_dir):
        self.save_dir  = Path(save_dir)
        self.best_dir  = self.save_dir / "best_model"
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.best_dir.mkdir(parents=True, exist_ok=True)
        self.meta_path  = self.save_dir / "meta.json"
        self.state_path = self.save_dir / "training_state.pt"

    # ─────────────────────────────────────────────────────────
    def save(self, model, tokenizer, optimizer, scheduler,
             scaler, epoch, best_f1, es_counter, thresholds=None):
        """Save current checkpoint (called every epoch)."""
        model.save_pretrained(self.save_dir)
        tokenizer.save_pretrained(self.save_dir)
        torch.save({
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler":    scaler.state_dict(),
        }, self.state_path)
        meta = {
            "epoch":      epoch,
            "best_f1":    float(best_f1),
            "es_counter": es_counter,
            "thresholds": thresholds or [0.5] * NUM_LABELS,
            "saved_at":   datetime.now().isoformat(),
        }
        self.meta_path.write_text(json.dumps(meta, indent=2))
        print(f"  💾 Checkpoint saved — epoch {epoch+1} | best F1 {best_f1:.4f}")

    # ─────────────────────────────────────────────────────────
    def save_best(self, model, tokenizer):
        """Called only when val F1 improves."""
        model.save_pretrained(self.best_dir)
        tokenizer.save_pretrained(self.best_dir)
        print(f"  🏆 Best model updated → {self.best_dir}")

    # ─────────────────────────────────────────────────────────
    def load(self, model, optimizer, scheduler, scaler):
        """Resume from the last checkpoint if it exists."""
        if not self.meta_path.exists():
            print("🆕 No checkpoint found — fresh training.")
            return 0, 0.0, 0, [0.5] * NUM_LABELS

        meta = json.loads(self.meta_path.read_text())

        # Load saved model weights into the (already‑initialised) model
        from transformers import AutoModelForSequenceClassification
        saved = AutoModelForSequenceClassification.from_pretrained(str(self.save_dir))
        model.load_state_dict(saved.state_dict(), strict=False)
        del saved

        ts = torch.load(self.state_path, map_location=DEVICE)
        optimizer.load_state_dict(ts["optimizer"])
        scheduler.load_state_dict(ts["scheduler"])
        scaler.load_state_dict(ts["scaler"])

        start  = meta["epoch"] + 1
        best   = meta["best_f1"]
        esc    = meta["es_counter"]
        thrs   = meta.get("thresholds", [0.5] * NUM_LABELS)
        print(f"✅ Resumed from epoch {start+1} | best F1 {best:.4f} | ES counter {esc}")
        return start, best, esc, thrs

    def model_exists(self):      return (self.save_dir / "config.json").exists()
    def best_exists(self):       return (self.best_dir  / "config.json").exists()


ckpt = CheckpointManager(CHECKPOINT_DIR)
print("CheckpointManager ready ✅")

## Section 5 — Data Download

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 5A — KAGGLE (Jigsaw Toxic Comments)
# ═══════════════════════════════════════════════════════════
from google.colab import files as colab_files

os.makedirs("/root/.kaggle", exist_ok=True)

kaggle_json = "/root/.kaggle/kaggle.json"

if not os.path.exists(kaggle_json):
    print("Upload your kaggle.json:")
    colab_files.upload()
    os.rename("kaggle.json", kaggle_json)
    os.chmod(kaggle_json, 600)

!pip install kaggle -q
!kaggle competitions download -c jigsaw-toxic-comment-classification-challenge \
    -p {DATA_DIR} -q

# Unzip outer archive
outer = f"{DATA_DIR}/jigsaw-toxic-comment-classification-challenge.zip"
with zipfile.ZipFile(outer) as z:
    z.extractall(DATA_DIR)

# Unzip any inner zips
for f in os.listdir(DATA_DIR):
    if f.endswith(".zip"):
        with zipfile.ZipFile(f"{DATA_DIR}/{f}") as z:
            z.extractall(DATA_DIR)

print("Jigsaw downloaded ✅")
print(os.listdir(DATA_DIR))

In [ ]:
# # ═══════════════════════════════════════════════════════════
# # SECTION 5B — HASOC 2021 (Hindi + Hinglish)
# # Tries GitHub first; falls back to synthetic Hinglish.
# # To use the real dataset: download manually from
# #   https://hasocfire.github.io/hasoc/2021/dataset.html
# # and upload hasoc_hi.tsv / hasoc_en.tsv to /content/data/
# # ═══════════════════════════════════════════════════════════
# import urllib.request

# HASOC_URLS = {
#     "en": ("https://raw.githubusercontent.com/Jeniya1378/HASOC-2021/"
#            "main/English/hasoc2021_en_train.tsv",
#            f"{DATA_DIR}/hasoc_en.tsv"),
#     "hi": ("https://raw.githubusercontent.com/Jeniya1378/HASOC-2021/"
#            "main/Hindi/hasoc2021_hi_train.tsv",
#            f"{DATA_DIR}/hasoc_hi.tsv"),
# }

# downloaded = {}
# for lang, (url, path) in HASOC_URLS.items():
#     # Check if manually uploaded first
#     if os.path.exists(path):
#         downloaded[lang] = path
#         print(f"✅ HASOC {lang.upper()} — found locally")
#         continue
#     try:
#         urllib.request.urlretrieve(url, path)
#         downloaded[lang] = path
#         print(f"✅ HASOC {lang.upper()} — downloaded from GitHub")
#     except Exception as e:
#         print(f"⚠️  HASOC {lang.upper()} — unavailable: {e}")

# print(f"\nAvailable HASOC languages: {list(downloaded.keys())}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 5B — HASOC 2019 (Hindi + English) — Real data, no registration
# Source: github.com/TharinduDR/HASOC-2019
# Note: These are the official test splits (train is password-protected)
# ═══════════════════════════════════════════════════════════
import os, urllib.request, pandas as pd

os.makedirs(DATA_DIR, exist_ok=True)

BASE = "https://raw.githubusercontent.com/TharinduDR/HASOC-2019/master/data"

HASOC_URLS = {
    "en": (f"{BASE}/hasoc2019_en_test.tsv", f"{DATA_DIR}/hasoc_en.tsv"),
    "hi": (f"{BASE}/hasoc2019_hi_test.tsv", f"{DATA_DIR}/hasoc_hi.tsv"),
}

downloaded = {}

for lang, (url, path) in HASOC_URLS.items():
    if os.path.exists(path):
        print(f"✅ HASOC 2019 {lang.upper()} — found locally")
        downloaded[lang] = path
        continue
    try:
        urllib.request.urlretrieve(url, path)
        print(f"✅ HASOC 2019 {lang.upper()} — downloaded from GitHub")
        downloaded[lang] = path
    except Exception as e:
        print(f"❌ HASOC 2019 {lang.upper()} — failed: {e}")

# ── Normalise columns to match HASOC 2021 schema ─────────
# 2019 columns: text_id | text | task_1 | task_2
# 2021 columns: text_id | text | task_1 | task_2  ← same, no changes needed

hasoc_dfs = {}

for lang, path in downloaded.items():
    df = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=["text_id", "text", "task_1", "task_2"],  # 2019 has no header row
        skiprows=0,
    )

    # Drop malformed rows
    df = df.dropna(subset=["text", "task_1"]).reset_index(drop=True)

    # Standardise labels (2019 uses same HOF/NOT + HATE/OFFN/PRFN/NONE)
    df["task_1"] = df["task_1"].str.strip().str.upper()
    df["task_2"] = df["task_2"].str.strip().str.upper()

    hasoc_dfs[lang] = df
    print(f"\n── HASOC 2019 {lang.upper()} ({len(df)} rows) ──")
    print(df[["text", "task_1", "task_2"]].head(5).to_string(index=False))

print(f"\nAvailable HASOC languages: {list(downloaded.keys())}")

# ── Label distribution ────────────────────────────────────
for lang, df in hasoc_dfs.items():
    print(f"\n[{lang.upper()}] task_1: {df['task_1'].value_counts().to_dict()}")
    print(f"[{lang.upper()}] task_2: {df['task_2'].value_counts().to_dict()}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 5C — LOAD & MERGE ALL DATASETS
# ═══════════════════════════════════════════════════════════

# ── Jigsaw ─────────────────────────────────────────────────
jigsaw             = pd.read_csv(f"{DATA_DIR}/train.csv")
jigsaw["source"]   = "jigsaw"
jigsaw["language"] = "en"

def load_hasoc(path, language):
    """Map HOF/NOT labels to the Jigsaw 6-label schema."""
    try:
        df = pd.read_csv(path, sep="\t", header=None, on_bad_lines="skip")
        # HASOC columns: id | text | label | sub_task_label
        df.columns = ["id", "comment_text", "label", "sub_label"][: len(df.columns)]
        df["comment_text"] = df["comment_text"].astype(str)
        df["toxic"]         = (df["label"] == "HOF").astype(int)
        df["severe_toxic"]  = 0
        df["obscene"]       = 0
        df["threat"]        = 0
        df["insult"]        = 0
        df["identity_hate"] = 0
        df["source"]        = "hasoc"
        df["language"]      = language
        return df[["comment_text"] + LABEL_COLS + ["source", "language"]]
    except Exception as e:
        print(f"  Could not parse {path}: {e}")
        return None

# ── HASOC ──────────────────────────────────────────────────
hasoc_frames = []
for lang, path in downloaded.items():
    df_h = load_hasoc(path, lang)
    if df_h is not None:
        hasoc_frames.append(df_h)
        print(f"  HASOC {lang.upper()}: {len(df_h):,} rows")

# ── Synthetic Hinglish fallback ────────────────────────────
if not hasoc_frames:
    print("\n⚠️  Using synthetic Hinglish sample (HASOC unavailable)")
    print("   For real results, download HASOC from the link above.\n")
    synth = {
        "comment_text": [
            "yaar tu bahut bura insaan hai bhai",
            "kya bakwas kar raha hai tu saale",
            "bhai teri aukaat kya hai bata",
            "abbe chup kar bakwaas band kar",
            "teri maa ki aankh sala",
            "bhai mast scene hai aaj yaar",
            "kya chal raha hai dost",
            "sab theek hai na yaar",
            "i hate people like you yaar bhai",
            "go to hell you stupid idiot bhai",
            "kya yaar tujhe kuch pata bhi hai",
            "bhai tu toh bilkul pagal hai",
            "awesome bhai bahut badhiya",
            "ek dum mast content hai bhai",
            "yaar ye toh ganda hai bilkul",
            "tujhe koi haq nahi bhai",
            "tu kuch bhi nahi hai yaar",
            "bhai great work kar raha hai tu",
            "sala haramkhor kahin ka",
            "bahut badiya dost shukriya",
        ],
        "toxic":         [0,1,1,1,1,0,0,0,1,1,0,0,0,0,1,1,1,0,1,0],
        "severe_toxic":  [0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0],
        "obscene":       [0,1,0,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0],
        "threat":        [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
        "insult":        [0,1,1,1,1,0,0,0,1,1,0,0,0,0,1,1,1,0,1,0],
        "identity_hate": [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
        "source":        ["hasoc_synthetic"] * 20,
        "language":      ["hi"] * 20,
    }
    hasoc_frames.append(pd.DataFrame(synth))

# ── Merge ──────────────────────────────────────────────────
jigsaw_sub = jigsaw[["comment_text"] + LABEL_COLS + ["source", "language"]].copy()
df_all     = pd.concat([jigsaw_sub] + hasoc_frames, ignore_index=True)
df_all     = df_all.dropna(subset=["comment_text"])
df_all["comment_text"] = df_all["comment_text"].astype(str)

print(f"\n📊 Total rows : {len(df_all):,}")
print(df_all["source"].value_counts().to_string())
print()
print(df_all["language"].value_counts().to_string())

## Section 6 — EDA

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 6 — EDA + LANGUAGE DISTRIBUTION
# ═══════════════════════════════════════════════════════════

df_all["text_len"] = df_all["comment_text"].str.split().str.len()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Exploratory Data Analysis", fontsize=15, fontweight="bold")

# Label counts
label_counts = df_all[LABEL_COLS].sum().sort_values(ascending=False)
bars = axes[0,0].bar(LABEL_DISPLAY, label_counts.values,
                     color="steelblue", edgecolor="white")
for bar, v in zip(bars, label_counts.values):
    axes[0,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                   f"{int(v):,}", ha="center", fontsize=8)
axes[0,0].set_title("Label Counts")
axes[0,0].set_xticklabels(LABEL_DISPLAY, rotation=30, ha="right")
axes[0,0].set_ylabel("Count")
axes[0,0].grid(axis="y", alpha=0.3)

# Language pie
lang_counts = df_all["language"].value_counts()
axes[0,1].pie(lang_counts.values, labels=lang_counts.index,
              autopct="%1.1f%%", colors=["steelblue","salmon","gold"])
axes[0,1].set_title("Language Distribution")

# Text length by language
for lang, grp in df_all.groupby("language"):
    axes[1,0].hist(grp["text_len"].clip(0, 250), bins=40,
                   alpha=0.6, label=lang, density=True)
axes[1,0].set_title("Text Length Distribution by Language")
axes[1,0].set_xlabel("Word count")
axes[1,0].set_ylabel("Density")
axes[1,0].legend()
axes[1,0].grid(alpha=0.3)

# Label co-occurrence heatmap
co = df_all[LABEL_COLS].astype(int).T.dot(df_all[LABEL_COLS].astype(int))
sns.heatmap(co, annot=True, fmt="d", ax=axes[1,1], cmap="Blues",
            xticklabels=LABEL_DISPLAY, yticklabels=LABEL_DISPLAY)
axes[1,1].set_title("Label Co-occurrence")
plt.setp(axes[1,1].get_xticklabels(), rotation=30, ha="right", fontsize=7)
plt.setp(axes[1,1].get_yticklabels(), fontsize=7)

plt.tight_layout()
plt.savefig(f"{DRIVE_ROOT}/eda.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n📏 Text length stats:")
print(df_all.groupby("language")["text_len"].describe().round(1).to_string())
print("\n📊 Label %:")
print((df_all[LABEL_COLS].mean() * 100).round(2).to_string())

## Section 7 — Preprocessing

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 7 — PREPROCESSING
# Hindi/Hinglish-aware text cleaning pipeline
# ═══════════════════════════════════════════════════════════
import emoji
from langdetect import detect, LangDetectException
from multiprocessing import Pool, cpu_count
from tqdm import tqdm

# ── Abbreviation dictionaries ──────────────────────────────
ENG_ABBREVS = {
    "wtf":"what the hell","stfu":"shut up","idk":"i don't know",
    "ngl":"not gonna lie","lmao":"laughing","omg":"oh my god",
    "u":"you","ur":"your","r":"are","bc":"because","tbh":"to be honest",
    "imo":"in my opinion","smh":"shaking my head","af":"very much",
    "ikr":"i know right","nvm":"never mind","fyi":"for your information",
}
HIN_ABBREVS = {
    "yaar":"friend","bhai":"brother","abbe":"hey",
    "sala":"jerk","saale":"jerk","bakwas":"nonsense",
    "chup":"silent","teri":"your","meri":"my",
    "aukaat":"worth","ganda":"dirty","bura":"bad",
    "mast":"awesome","bc":"son of a bitch",
    "bsdk":"son of a bitch","mc":"son of a bitch",
    "kya":"what","toh":"then","nahi":"no","hai":"is",
}

def detect_lang(text):
    try:
        lang = detect(str(text)[:300])
        return lang if lang in ("en","hi") else "en"
    except LangDetectException:
        return "en"

def clean_text(text):
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def handle_emojis(text):
    return emoji.demojize(text, delimiters=(" ", " "))

def norm_repeated(text):
    return re.sub(r"(.)\1{2,}", r"\1\1", text)

def expand_abbrevs(text, lang="en"):
    abbrevs = dict(ENG_ABBREVS)
    if lang in ("hi", "hi-en"):
        abbrevs.update(HIN_ABBREVS)
    tokens = text.lower().split()
    return " ".join(abbrevs.get(t, t) for t in tokens)

def full_pipeline(args):
    """Args = (text, lang) — unpacked for multiprocessing."""
    text, lang = args
    text = clean_text(str(text))
    text = handle_emojis(text)
    text = norm_repeated(text)
    text = expand_abbrevs(text, lang)
    return text.lower().strip()

# ── Apply in parallel ──────────────────────────────────────
print(f"CPU cores: {cpu_count()}")
args = list(zip(df_all["comment_text"].tolist(),
                df_all["language"].tolist()))

with Pool(cpu_count()) as pool:
    cleaned = list(tqdm(pool.imap(full_pipeline, args),
                        total=len(df_all), desc="Preprocessing"))

df_all["clean_text"] = cleaned

print("\nSample outputs:")
for _, row in df_all.sample(4, random_state=42).iterrows():
    print(f"  [{row['language']}] IN : {row['comment_text'][:80]}")
    print(f"         OUT: {row['clean_text'][:80]}\n")

## Section 8 — Train / Val Split

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 8 — TRAIN / VAL SPLIT  (stratified)
# ═══════════════════════════════════════════════════════════
from sklearn.model_selection import train_test_split

df_all["any_toxic"] = (df_all[LABEL_COLS].sum(axis=1) > 0).astype(int)

train_df, val_df = train_test_split(
    df_all, test_size=0.1, random_state=SEED,
    stratify=df_all["any_toxic"]
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"Train : {len(train_df):,}")
print(f"Val   : {len(val_df):,}")
print("\nTrain label %:")
print((train_df[LABEL_COLS].mean() * 100).round(2).to_string())
print("\nTrain language split:")
print(train_df["language"].value_counts().to_string())

## Section 9 — Dataset + DataLoaders

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 9 — DATASET + TOKENIZER
# ═══════════════════════════════════════════════════════════
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

print("Loading tokenizer…")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class AbuseDataset(Dataset):
    def __init__(self, df, max_len=MAX_LEN):
        self.texts  = df["clean_text"].tolist()
        self.labels = df[LABEL_COLS].values.astype(np.float32)
        self.langs  = df["language"].tolist()
        self.max_len = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length     = self.max_len,
            padding        = "max_length",
            truncation     = True,
            return_tensors = "pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.float),
            "lang":           self.langs[idx],
        }

train_ds = AbuseDataset(train_df)
val_ds   = AbuseDataset(val_df)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          pin_memory=True, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          pin_memory=True, num_workers=2)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Sample shape  : {train_ds[0]['input_ids'].shape}")

## Section 10 — Model

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 10 — MODEL
# Loads from Drive if available, else from HuggingFace.
# ═══════════════════════════════════════════════════════════
from transformers import (AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)

def build_model(source_dir=None):
    src = str(source_dir) if source_dir else MODEL_NAME
    m   = AutoModelForSequenceClassification.from_pretrained(
        src,
        num_labels    = NUM_LABELS,
        problem_type  = "multi_label_classification",
        ignore_mismatched_sizes = True,
    )
    return m.to(DEVICE)

if ckpt.best_exists():
    print("Loading BEST MODEL from Drive…")
    model = build_model(ckpt.best_dir)
elif ckpt.model_exists():
    print("Loading CHECKPOINT from Drive…")
    model = build_model(ckpt.save_dir)
else:
    print("Loading pretrained from HuggingFace…")
    model = build_model()

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")

# ── Per-label class weights ─────────────────────────────────
pos_weights = []
for col in LABEL_COLS:
    pos = max(train_df[col].sum(), 1)
    neg = len(train_df) - pos
    pos_weights.append(neg / pos)
pos_w_tensor = torch.tensor(pos_weights, dtype=torch.float).to(DEVICE)
criterion    = nn.BCEWithLogitsLoss(pos_weight=pos_w_tensor)

print("\nClass weights:")
for col, w in zip(LABEL_COLS, pos_weights):
    print(f"  {col:<15} {w:.1f}×")

# ── Optimizer + Scheduler ───────────────────────────────────
optimizer = torch.optim.AdamW(model.parameters(),
                               lr=LR, weight_decay=WEIGHT_DECAY)
total_optim_steps = (len(train_loader) // GRAD_ACCUM_STEPS) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = WARMUP_STEPS,
    num_training_steps = total_optim_steps,
)
scaler = torch.cuda.amp.GradScaler()
print("\nOptimizer, scheduler, scaler ready ✅")

## Section 11 — Training

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 11 — TRAINING LOOP
# Features:
#   • Resume from Drive checkpoint automatically
#   • Gradient accumulation (effective batch = 32)
#   • Mixed-precision AMP (2-3× faster on T4)
#   • Early stopping — patience = 5
#   • Saves best model to Drive on improvement
# ═══════════════════════════════════════════════════════════
from sklearn.metrics import f1_score

# Resume state if checkpoint exists
start_epoch, best_f1, es_counter, best_thresholds = ckpt.load(
    model, optimizer, scheduler, scaler
)
model = model.to(DEVICE)

print(f"\nStarting from epoch {start_epoch + 1}/{EPOCHS}")
print(f"Early stop patience: {EARLY_STOP_PATIENCE}")
print(f"Grad accum steps: {GRAD_ACCUM_STEPS} (effective batch={BATCH_SIZE*GRAD_ACCUM_STEPS})\n")

for epoch in range(start_epoch, EPOCHS):

    # ── TRAIN ────────────────────────────────────────────────
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                desc=f"Ep {epoch+1}/{EPOCHS} [Train]")

    for step, batch in pbar:
        ids   = batch["input_ids"].to(DEVICE, non_blocking=True)
        mask  = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labs  = batch["labels"].to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast():
            out  = model(input_ids=ids, attention_mask=mask)
            loss = criterion(out.logits, labs) / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        real_loss = loss.item() * GRAD_ACCUM_STEPS
        total_loss += real_loss
        pbar.set_postfix({"loss": f"{real_loss:.4f}"})

        if step % 100 == 0:
            wandb.log({"step_loss": real_loss,
                       "step": epoch * len(train_loader) + step})

    avg_loss = total_loss / len(train_loader)

    # ── VALIDATE ─────────────────────────────────────────────
    model.eval()
    all_probs, all_labels = [], []

    vbar = tqdm(val_loader, desc=f"Ep {epoch+1}/{EPOCHS} [Val]  ")
    with torch.no_grad():
        for batch in vbar:
            ids   = batch["input_ids"].to(DEVICE, non_blocking=True)
            mask  = batch["attention_mask"].to(DEVICE, non_blocking=True)
            labs  = batch["labels"]
            with torch.cuda.amp.autocast():
                out = model(input_ids=ids, attention_mask=mask)
            all_probs.extend(torch.sigmoid(out.logits).cpu().numpy())
            all_labels.extend(labs.numpy())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_preds  = (all_probs > 0.5).astype(int)

    val_f1 = f1_score(all_labels.astype(int), all_preds,
                      average="macro", zero_division=0)

    # Per-label F1 for W&B
    per_label_f1 = f1_score(all_labels.astype(int), all_preds,
                             average=None, zero_division=0)
    wandb.log({
        "epoch":      epoch + 1,
        "train_loss": avg_loss,
        "val_f1_macro": val_f1,
        **{f"f1_{col}": float(f) for col, f in zip(LABEL_COLS, per_label_f1)},
    })

    # ── Status print ─────────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"  Epoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss : {avg_loss:.4f}")
    print(f"  Val F1     : {val_f1:.4f}  (best so far: {best_f1:.4f})")
    print(f"  ES counter : {es_counter}/{EARLY_STOP_PATIENCE}")
    print(f"{'='*55}")
    for col, f in zip(LABEL_COLS, per_label_f1):
        print(f"    {col:<15} F1={f:.4f}")

    # ── Save / Early stop ────────────────────────────────────
    if val_f1 > best_f1:
        best_f1    = val_f1
        es_counter = 0
        ckpt.save_best(model, tokenizer)
        ckpt.save(model, tokenizer, optimizer, scheduler, scaler,
                  epoch, best_f1, es_counter)
        print(f"\n  🏆 New best! F1={best_f1:.4f} — saved to Drive")
    else:
        es_counter += 1
        ckpt.save(model, tokenizer, optimizer, scheduler, scaler,
                  epoch, best_f1, es_counter)
        print(f"\n  No improvement. ES counter={es_counter}/{EARLY_STOP_PATIENCE}")
        if es_counter >= EARLY_STOP_PATIENCE:
            print("\n🛑 EARLY STOPPING triggered.")
            break

wandb.finish()
print(f"\n🎉 Training done! Best Val F1 = {best_f1:.4f}")

## Section 12 — Load Best Model for Evaluation

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 12 — LOAD BEST MODEL FOR EVALUATION
# ═══════════════════════════════════════════════════════════
from transformers import AutoModelForSequenceClassification, AutoTokenizer as AT

print("Loading best model…")
eval_model = AutoModelForSequenceClassification.from_pretrained(
    str(ckpt.best_dir),
    attn_implementation="eager"    # ← required for attention visualization
).to(DEVICE)
eval_tokenizer = AT.from_pretrained(str(ckpt.best_dir))
eval_model.eval()
print("Best model loaded ✅")

def get_val_preds(model, loader):
    all_probs, all_labels, all_langs = [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Predicting val set"):
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            with torch.cuda.amp.autocast():
                out = model(input_ids=ids, attention_mask=mask)
            all_probs.extend(torch.sigmoid(out.logits).cpu().numpy())
            all_labels.extend(batch["labels"].numpy())
            all_langs.extend(batch["lang"])
    return np.array(all_probs), np.array(all_labels), all_langs

val_probs, val_labels, val_langs = get_val_preds(eval_model, val_loader)
print(f"Predictions shape : {val_probs.shape}")

## Section 13 — Threshold Tuning

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 13 — THRESHOLD TUNING
# Finds the optimal decision threshold per label on the
# validation set — big F1 gains vs fixed 0.5.
# ═══════════════════════════════════════════════════════════
from sklearn.metrics import f1_score as sk_f1

search_thresholds = np.arange(0.05, 0.95, 0.025)
best_thresholds   = []

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("Threshold Tuning per Label", fontsize=13, fontweight="bold")

for i, (col, label) in enumerate(zip(LABEL_COLS, LABEL_DISPLAY)):
    ax  = axes[i // 3][i % 3]
    f1s = []

    for t in search_thresholds:
        preds = (val_probs[:, i] > t).astype(int)
        f1s.append(sk_f1(val_labels[:, i].astype(int), preds, zero_division=0))

    best_t = float(search_thresholds[int(np.argmax(f1s))])
    best_thresholds.append(best_t)

    ax.plot(search_thresholds, f1s, "b-", linewidth=1.5)
    ax.axvline(best_t, color="red", linestyle="--", linewidth=1.5,
               label=f"Best t={best_t:.3f} | F1={max(f1s):.3f}")
    ax.scatter([best_t], [max(f1s)], color="red", s=60, zorder=5)
    ax.set_title(label, fontweight="bold")
    ax.set_xlabel("Threshold")
    ax.set_ylabel("F1 Score")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{DRIVE_ROOT}/threshold_tuning.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n🎯 Optimal thresholds:")
for col, t in zip(LABEL_COLS, best_thresholds):
    print(f"  {col:<18} → {t:.3f}")

# Persist to Drive checkpoint meta
meta = json.loads(ckpt.meta_path.read_text())
meta["thresholds"] = best_thresholds
ckpt.meta_path.write_text(json.dumps(meta, indent=2))
print("\nThresholds saved to Drive ✅")

## Section 14 — Full Evaluation Suite

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 14A — PER-LABEL METRICS TABLE
# ═══════════════════════════════════════════════════════════
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_recall_curve,
    roc_curve, average_precision_score,
    matthews_corrcoef, accuracy_score, hamming_loss,
)

# Apply tuned thresholds
val_preds = np.zeros_like(val_probs)
for i, t in enumerate(best_thresholds):
    val_preds[:, i] = (val_probs[:, i] > t).astype(int)

val_preds_def = (val_probs > 0.5).astype(int)

rows = []
for i, (col, label) in enumerate(zip(LABEL_COLS, LABEL_DISPLAY)):
    yt, yp, ypr = (val_labels[:, i].astype(int),
                   val_preds[:, i].astype(int),
                   val_probs[:, i])
    try:
        auc = roc_auc_score(yt, ypr)
        ap  = average_precision_score(yt, ypr)
    except:
        auc = ap = 0.0
    try:
        mcc = matthews_corrcoef(yt, yp)
    except:
        mcc = 0.0
    rows.append({
        "Label":       label,
        "F1-binary":   round(sk_f1(yt, yp, average="binary",   zero_division=0), 4),
        "F1-macro":    round(sk_f1(yt, yp, average="macro",    zero_division=0), 4),
        "F1-weighted": round(sk_f1(yt, yp, average="weighted", zero_division=0), 4),
        "ROC-AUC":     round(auc,  4),
        "Avg-Prec":    round(ap,   4),
        "MCC":         round(mcc,  4),
        "Threshold":   round(best_thresholds[i], 3),
    })

mdf = pd.DataFrame(rows)
print("=" * 80)
print("PER-LABEL METRICS (tuned thresholds)")
print("=" * 80)
print(mdf.to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 14B — OVERALL MULTI-LABEL METRICS
# ═══════════════════════════════════════════════════════════

yt_int = val_labels.astype(int)
yp_int = val_preds.astype(int)

print("\n" + "=" * 60)
print("OVERALL MULTI-LABEL METRICS")
print("=" * 60)
overall_metrics = {
    "Hamming Loss":             round(hamming_loss(yt_int, yp_int), 4),
    "Subset Accuracy":          round(accuracy_score(yt_int, yp_int), 4),
    "Macro  F1 (tuned)":        round(sk_f1(yt_int, yp_int, average="macro",    zero_division=0), 4),
    "Micro  F1 (tuned)":        round(sk_f1(yt_int, yp_int, average="micro",    zero_division=0), 4),
    "Weighted F1 (tuned)":      round(sk_f1(yt_int, yp_int, average="weighted", zero_division=0), 4),
    "Macro  F1 (default 0.5)":  round(sk_f1(yt_int, val_preds_def.astype(int), average="macro", zero_division=0), 4),
}
for k, v in overall_metrics.items():
    bar  = "█" * int(v * 20)
    print(f"  {k:<30}: {v:.4f}  [{bar:<20}]")

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 14C — ROC + PRECISION-RECALL CURVES
# ═══════════════════════════════════════════════════════════
colors = ["steelblue","firebrick","seagreen","darkorange","purple","sienna"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, (col, label) in enumerate(zip(LABEL_COLS, LABEL_DISPLAY)):
    yt  = val_labels[:, i].astype(int)
    ypr = val_probs[:, i]
    if yt.sum() == 0: continue

    fpr, tpr, _ = roc_curve(yt, ypr)
    auc = roc_auc_score(yt, ypr)
    axes[0].plot(fpr, tpr, color=colors[i], label=f"{label} ({auc:.3f})")

    prec_c, rec_c, _ = precision_recall_curve(yt, ypr)
    ap = average_precision_score(yt, ypr)
    axes[1].plot(rec_c, prec_c, color=colors[i], label=f"{label} ({ap:.3f})")

for ax, title, xl, yl in [
    (axes[0], "ROC Curves (AUC)", "FPR", "TPR"),
    (axes[1], "Precision-Recall Curves (AP)", "Recall", "Precision"),
]:
    axes[0].plot([0,1],[0,1],"k--",alpha=0.4)
    ax.set_title(title); ax.set_xlabel(xl); ax.set_ylabel(yl)
    ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{DRIVE_ROOT}/roc_pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 14D — CONFUSION MATRICES
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("Confusion Matrices — Tuned Thresholds", fontsize=13, fontweight="bold")

for i, (col, label) in enumerate(zip(LABEL_COLS, LABEL_DISPLAY)):
    ax = axes[i // 3][i % 3]
    yt = val_labels[:, i].astype(int)
    yp = val_preds[:, i].astype(int)
    cm = confusion_matrix(yt, yp)
    sns.heatmap(cm, annot=True, fmt="d", ax=ax, cmap="Blues",
                xticklabels=["Clean","Toxic"],
                yticklabels=["Clean","Toxic"])
    tn, fp, fn, tp = cm.ravel()
    tpr = tp / max(tp + fn, 1)
    fpr = fp / max(fp + tn, 1)
    ax.set_title(f"{label}\nTPR={tpr:.3f}  FPR={fpr:.3f}")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")

plt.tight_layout()
plt.savefig(f"{DRIVE_ROOT}/confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 14E — PER-LANGUAGE BREAKDOWN
# ═══════════════════════════════════════════════════════════
val_lang_arr = np.array(val_langs)
unique_langs = [l for l in np.unique(val_lang_arr)
                if (val_lang_arr == l).sum() >= 10]

lang_summary = {}
print("=" * 55)
print("PER-LANGUAGE EVALUATION")
print("=" * 55)

for lang in unique_langs:
    mask = val_lang_arr == lang
    lp   = val_probs[mask]
    ll   = val_labels[mask]
    lpred = np.zeros_like(lp)
    for i, t in enumerate(best_thresholds):
        lpred[:, i] = (lp[:, i] > t).astype(int)

    mac  = sk_f1(ll.astype(int), lpred.astype(int), average="macro",    zero_division=0)
    mic  = sk_f1(ll.astype(int), lpred.astype(int), average="micro",    zero_division=0)
    wt   = sk_f1(ll.astype(int), lpred.astype(int), average="weighted", zero_division=0)
    hl   = hamming_loss(ll.astype(int), lpred.astype(int))
    try:
        auc = roc_auc_score(ll.astype(int), lp, average="macro",
                            multi_class="ovr", labels=[0,1])
    except:
        auc = float("nan")

    lang_summary[lang] = {"n": int(mask.sum()), "macro_f1": mac, "micro_f1": mic}
    print(f"\n  [{lang.upper()}]  n={mask.sum():,}")
    print(f"    Macro  F1 : {mac:.4f}")
    print(f"    Micro  F1 : {mic:.4f}")
    print(f"    Weighted F1 : {wt:.4f}")
    print(f"    Hamming Loss : {hl:.4f}")
    print(f"    ROC-AUC     : {auc:.4f}" if not np.isnan(auc) else "    ROC-AUC     : n/a")

if len(lang_summary) > 1:
    fig, ax = plt.subplots(figsize=(7, 4))
    langs_l  = list(lang_summary.keys())
    mac_f1s  = [lang_summary[l]["macro_f1"] for l in langs_l]
    bars = ax.bar(langs_l, mac_f1s,
                  color=["steelblue","salmon","gold"][:len(langs_l)],
                  edgecolor="white", width=0.4)
    for b, v in zip(bars, mac_f1s):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.008,
                f"{v:.3f}", ha="center", fontweight="bold")
    ax.set_title("Macro F1 by Language")
    ax.set_ylim(0, 1); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{DRIVE_ROOT}/per_language_f1.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 14F — CALIBRATION PLOTS
# "If the model says 80% confidence, is it right 80% of
#  the time?" — perfect calibration = diagonal line.
# ═══════════════════════════════════════════════════════════
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("Reliability / Calibration Diagrams", fontsize=13, fontweight="bold")

for i, (col, label) in enumerate(zip(LABEL_COLS, LABEL_DISPLAY)):
    ax = axes[i // 3][i % 3]
    yt = val_labels[:, i].astype(int)
    yp = val_probs[:, i]
    if yt.sum() < 10:
        ax.set_title(f"{label}\n(too few positives)"); continue
    try:
        frac, mean_pred = calibration_curve(yt, yp, n_bins=10, strategy="quantile")
        ax.plot(mean_pred, frac, "s-b", linewidth=1.5, label="Model")
        ax.plot([0,1],[0,1],"k--", label="Perfect")
        ax.fill_between(mean_pred, frac, mean_pred, alpha=0.15, color="red")
    except Exception as e:
        ax.text(0.5, 0.5, str(e), ha="center", transform=ax.transAxes)
    ax.set_title(label); ax.set_xlabel("Mean predicted prob")
    ax.set_ylabel("Fraction of positives")
    ax.legend(fontsize=7); ax.grid(alpha=0.3); ax.set_xlim(0,1); ax.set_ylim(0,1)

plt.tight_layout()
plt.savefig(f"{DRIVE_ROOT}/calibration.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 14G — CONFIDENCE HISTOGRAM
# Shows how spread-out the model's confidence is.
# Well-separated blue/red peaks → good discrimination.
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("Confidence Distribution per Label", fontsize=13, fontweight="bold")

for i, (col, label) in enumerate(zip(LABEL_COLS, LABEL_DISPLAY)):
    ax = axes[i // 3][i % 3]
    yt = val_labels[:, i].astype(int)
    yp = val_probs[:, i]
    ax.hist(yp[yt == 0], bins=40, alpha=0.6, color="steelblue",
            label="Clean", density=True)
    ax.hist(yp[yt == 1], bins=40, alpha=0.6, color="firebrick",
            label="Toxic", density=True)
    ax.axvline(best_thresholds[i], color="black", linestyle="--",
               linewidth=1.5, label=f"Threshold={best_thresholds[i]:.2f}")
    ax.set_title(label); ax.set_xlabel("Predicted probability")
    ax.set_ylabel("Density"); ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(f"{DRIVE_ROOT}/confidence_hist.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 14H — ERROR ANALYSIS
# Top false positives, false negatives, and confident TPs
# ═══════════════════════════════════════════════════════════
val_text_arr = np.array(val_df["comment_text"].tolist())

for focus_idx, focus_label in [(0, "Toxic"), (4, "Insult")]:
    yt   = val_labels[:, focus_idx].astype(int)
    yp   = val_preds[:, focus_idx].astype(int)
    yprb = val_probs[:, focus_idx]

    fp_mask = (yp == 1) & (yt == 0)
    fn_mask = (yp == 0) & (yt == 1)
    tp_mask = (yp == 1) & (yt == 1)

    print(f"\n{'='*60}")
    print(f"  ERROR ANALYSIS — {focus_label}  (threshold={best_thresholds[focus_idx]:.3f})")
    print(f"{'='*60}")
    print(f"  FP={fp_mask.sum()} | FN={fn_mask.sum()} | TP={tp_mask.sum()}")

    print(f"\n  🔴 TOP 5 FALSE POSITIVES (predicted {focus_label}, actually clean):")
    fps = sorted(zip(yprb[fp_mask], val_text_arr[fp_mask]), reverse=True)[:5]
    for prob, text in fps:
        print(f"    [{prob:.3f}] {text[:110]}")

    print(f"\n  🟡 TOP 5 FALSE NEGATIVES (missed {focus_label}):")
    fns = sorted(zip(yprb[fn_mask], val_text_arr[fn_mask]))[:5]
    for prob, text in fns:
        print(f"    [{prob:.3f}] {text[:110]}")

    print(f"\n  🟢 TOP 3 CONFIDENT TRUE POSITIVES:")
    tps = sorted(zip(yprb[tp_mask], val_text_arr[tp_mask]), reverse=True)[:3]
    for prob, text in tps:
        print(f"    [{prob:.3f}] {text[:110]}")

## Section 15 — Attention Visualization

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 15 — ATTENTION VISUALIZATION
# Extracts CLS-token attention from the last transformer
# layer to show which words drove each prediction.
# ═══════════════════════════════════════════════════════════

def predict_with_attention(text, model=eval_model, tok=eval_tokenizer):
    """Returns probs, word-level attention weights, and tokens."""
    cleaned = full_pipeline((text, detect_lang(text)))
    inputs  = tok(
        cleaned,
        return_tensors = "pt",
        max_length     = MAX_LEN,
        truncation     = True,
        padding        = True,
    ).to(DEVICE)

    with torch.no_grad():
        out = model(**inputs, output_attentions=True)

    probs   = torch.sigmoid(out.logits[0]).cpu().numpy()
    # Last layer, average over all heads, CLS row → shape (seq_len,)
    attn    = out.attentions[-1][0].mean(0)[0].cpu().numpy()
    real_n  = int((inputs["input_ids"][0] != tok.pad_token_id).sum())
    tokens  = tok.convert_ids_to_tokens(inputs["input_ids"][0])[:real_n]
    attn    = attn[:real_n]
    attn    = attn / (attn.max() + 1e-9)

    # Merge sub-word tokens → readable words
    words, wts = [], []
    for tok_s, w in zip(tokens, attn):
        clean_t = tok_s.replace("▁","").replace("##","")
        if clean_t in ("[CLS]","[SEP]","[PAD]","","<s>","</s>"): continue
        if words and (tok_s.startswith("##") or tok_s.startswith("▁") is False
                      and not tok_s.startswith("[") and words):
            # try to merge sub-words
            words[-1] += clean_t
            wts[-1]    = max(wts[-1], w)
        else:
            words.append(clean_t)
            wts.append(float(w))

    return probs, words[:20], np.array(wts[:20])


def plot_prediction(text):
    probs, words, wts = predict_with_attention(text)
    lang = detect_lang(text)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4),
                                    gridspec_kw={"width_ratios": [3, 1]})
    fig.suptitle(f'Input: "{text[:80]}{"…" if len(text)>80 else ""}"  [{lang.upper()}]',
                 fontsize=10, style="italic")

    # Attention bar chart
    cmap = plt.cm.Reds
    bar_colors = cmap(wts / (wts.max() + 1e-9))
    ax1.bar(range(len(words)), wts, color=bar_colors, edgecolor="white")
    ax1.set_xticks(range(len(words)))
    ax1.set_xticklabels(words, rotation=40, ha="right", fontsize=9)
    ax1.set_ylabel("Attention (normalised)")
    ax1.set_title("Word Attention — Last Layer, Avg Heads")
    ax1.grid(axis="y", alpha=0.3)

    # Label probability bars
    clrs = ["firebrick" if p > t else "steelblue"
            for p, t in zip(probs, best_thresholds)]
    bars = ax2.barh(LABEL_DISPLAY, probs, color=clrs, edgecolor="white")
    for i, (b, p, t) in enumerate(zip(bars, probs, best_thresholds)):
        ax2.axvline(t, color="gray", linestyle=":", linewidth=0.8)
        ax2.text(min(p + 0.02, 0.93), i, f"{p:.2f}",
                 va="center", fontsize=8,
                 color="white" if p > 0.65 else "black", fontweight="bold")
    ax2.set_xlim(0, 1)
    ax2.set_title("Label Probabilities\n(red = above threshold)")
    ax2.grid(axis="x", alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("\nVerdict:")
    flagged = []
    for label, p, t in zip(LABEL_DISPLAY, probs, best_thresholds):
        icon = "🔴" if p > t else "🟢"
        print(f"  {icon} {label:<15} {p:.3f}  (t={t:.2f})")
        if p > t: flagged.append(label)
    if flagged:
        print(f"\n  ⚠️  FLAGGED: {', '.join(flagged)}")
    else:
        print("\n  ✅ CLEAN")
    return probs

# ── Example calls ──────────────────────────────────────────
for ex in [
    "you are such a complete moron wtf is wrong with you",
    "yaar tu bahut bura insaan hai bhai seriously",
    "I hope you have an absolutely wonderful day!",
]:
    print(f"\n{'─'*55}")
    plot_prediction(ex)

## Section 16 — Interactive Demo

In [ ]:
# ═══════════════════════════════════════════════════════════
# SECTION 16 — INTERACTIVE DEMO
# Type any sentence (English or Hinglish) and get:
#   • Detected language
#   • Per-label probability + verdict
#   • Confidence bar chart
#   • Word attention heatmap
# ═══════════════════════════════════════════════════════════
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Widgets ───────────────────────────────────────────────
title = widgets.HTML(
    "<h3 style='color:#2c3e50'>🛡️ Multilingual Abuse Detector</h3>"
    "<p style='color:#7f8c8d'>Supports English &amp; Hinglish</p>"
)
text_box = widgets.Textarea(
    placeholder = "Type your sentence here…",
    layout      = widgets.Layout(width="100%", height="90px"),
)
btn      = widgets.Button(description="🔍  Analyse",
                          button_style="primary",
                          layout=widgets.Layout(width="140px", height="38px"))
out      = widgets.Output()

# ── Analysis callback ─────────────────────────────────────
def on_analyse(b):
    with out:
        clear_output(wait=True)
        text = text_box.value.strip()
        if not text:
            print("⚠️  Please enter some text first."); return

        lang    = detect_lang(text)
        cleaned = full_pipeline((text, lang))

        print(f"📝  Input   : {text}")
        print(f"🌐  Language: {lang.upper()}")
        print(f"🧹  Cleaned : {cleaned}")
        print()

        # Predict
        probs, words, wts = predict_with_attention(text)

        # ── Results table ─────────────────────────────────
        print(f"{'─'*54}")
        print(f"{'Label':<18} {'Prob':>5}  {'Bar':^22}  Verdict")
        print(f"{'─'*54}")
        flagged = []
        for label, prob, thresh in zip(LABEL_DISPLAY, probs, best_thresholds):
            filled = int(prob * 20)
            bar    = "█" * filled + "░" * (20 - filled)
            flag   = "🔴 FLAGGED" if prob > thresh else "🟢 clean  "
            print(f"{label:<18} {prob:>4.1%}  [{bar}]  {flag}")
            if prob > thresh: flagged.append(label)
        print(f"{'─'*54}")

        if flagged:
            print(f"\n  ⚠️  VERDICT: Harmful content detected → {', '.join(flagged)}")
        else:
            print("\n  ✅ VERDICT: Text appears clean.")

        # ── Mini attention chart ──────────────────────────
        if words:
            fig, ax = plt.subplots(figsize=(12, 2.5))
            norm_wts = wts / (wts.max() + 1e-9)
            ax.bar(words, norm_wts, color=plt.cm.Reds(norm_wts), edgecolor="white")
            ax.set_title("Word Attention — which tokens influenced the prediction")
            ax.set_ylabel("Attention")
            ax.set_xticklabels(words, rotation=35, ha="right", fontsize=9)
            ax.grid(axis="y", alpha=0.3)
            plt.tight_layout(); plt.show()

btn.on_click(on_analyse)

display(widgets.VBox([
    title,
    widgets.Label("Enter text:"),
    text_box,
    btn,
    out,
], layout=widgets.Layout(padding="16px", border="1px solid #ddd",
                          border_radius="8px", width="800px")))